# Gauge Equivariance Test

게이지 변환 $U' = g U g^\dagger$ 에 대해 레이어가 equivariant한지 확인:  
$g \cdot \text{Layer}(x, U) \stackrel{?}{=} \text{Layer}(g \cdot x, U')$  
norm2(A - B) ≈ 0 이면 equivariant.

In [1]:
using LatticeDiracOperators
using Gaugefields
using Wilsonloop
using LMP
using LinearAlgebra

┌ Warning: Circular dependency detected. Precompilation will be skipped for:
│   ChainRules [082447d4-558c-5d27-93f4-14fc19e9eca2]
│   SpecialFunctionsExt [1285c0f1-ff9f-5867-b66e-0f359bcf09ba]
│   GPUArrays [0c68f7d7-f131-5f86-a1c3-88cf8149b2d7]
│   LMP [a75e356b-bbea-40ef-9c9d-31e1efba0018]
│   KernelAbstractions [63c18a36-062a-441e-b654-da1e3ab1ce7c]
│   CUDA [052768ef-5323-5732-b1bb-66c8b64840ba]
│   SparseArraysExt [85068d23-b5fb-53f1-8204-05c2aba6942f]
│   StructArraysGPUArraysCoreExt [cbbf52aa-d660-55d4-8a5a-33845bbaa85d]
│   AtomixCUDAExt [13011619-4c7c-5ef0-948f-5fc81565cd05]
│   LinearAlgebraExt [66d79d19-2cc4-5b0b-ac7a-b340256d1ecd]
│   ZygoteColorsExt [e68c091a-8ea5-5ca7-be4f-380657d4ad79]
│   ChainRulesCoreExt [eae2faf6-b232-58cb-a410-7764fda2830c]
│   Zygote [e88e6eb3-aa80-5325-afca-941959d7151f]
└ @ Pkg.API C:\Users\_s2220186\AppData\Local\Programs\Julia-1.10.7\share\julia\stdlib\v1.10\Pkg\src\API.jl:1279
[ Info: Precompiling LMP [a75e356b-bbea-40ef-9c9d-31e1efba0018]
[ 

## 1. 격자/게이지 설정

In [2]:
NX = 4; NY = 4; NZ = 4; NT = 4
Nwing = 1; Dim = 4; NC = 3

U = Initialize_4DGaugefields(NC, Nwing, NX, NY, NZ, NT, condition="hot")

x = Initialize_pseudofermion_fields(U[1], "Wilson", nowing=true)
gauss_distribution_fermion!(x)

params = Dict()
params["Dirac_operator"] = "Wilson"
params["κ"] = 0.163
params["eps_CG"] = 1.0e-8
params["verbose_level"] = 3

D = Dirac_operator(U, x, params)
println("Setup complete")

┌ Warning: Now, Nwing=0 is recommended. Please use NDW = 0 like Initialize_Gaugefields(NC,0,NX,NY,NZ,NT). 
└ @ Gaugefields.AbstractGaugefields_module C:\Users\_s2220186\.julia\packages\Gaugefields\JBcsF\src\AbstractGaugefields.jl:371


Setup complete


## 2. 게이지 변환 준비

In [3]:
# 랜덤 게이지 변환 g
G = Initialize_4DGaugefields(NC, Nwing, NX, NY, NZ, NT, condition="hot")
g = G[1]  # 사이트별 SU(3) 행렬
temp = deepcopy(G[2])  # 작업용 버퍼

# 변환된 게이지 U' = g U g†
V = deepcopy(U)
gUgshift!(V, g, temp)

# 변환된 페르미온 x' = g x
gx = g * x

# 변환된 Dirac operator
D_transformed = Dirac_operator(V, gx, params)
println("Gauge transform ready")

┌ Warning: Now, Nwing=0 is recommended. Please use NDW = 0 like Initialize_Gaugefields(NC,0,NX,NY,NZ,NT). 
└ @ Gaugefields.AbstractGaugefields_module C:\Users\_s2220186\.julia\packages\Gaugefields\JBcsF\src\AbstractGaugefields.jl:371


Gauge transform ready


## 3. PTLayer Equivariance Test

In [4]:
P = Vector{Paths}(undef, 9)
for i in 1:9
    if i <= 5
        P[i] = Paths([[i-1]])
    else
        P[i] = Paths([[-(i-5)]])
    end
end

# 9개 입력 페르미온 생성 (paths 수와 일치해야 함)
xs = [Initialize_pseudofermion_fields(U[1], "Wilson", nowing=true) for _ in 1:9]
for xi in xs; gauss_distribution_fermion!(xi); end
gxs = [g * xi for xi in xs]

pt = PTLayer(P, U, 2)
pt_transformed = PTLayer(P, V, 2)
# 같은 W를 공유
pt_transformed.W = deepcopy(pt.W)

# Path A: Layer(xs, U) → g * result
out_A = pt(xs)
A = [g * out_A[i] for i in eachindex(out_A)]

# Path B: Layer(g*xs, U')
B = pt_transformed(gxs)

err = sum(norm2(A[i] - B[i]) for i in eachindex(A))
println("PTLayer equivariance error: ", err)

PTLayer equivariance error: 6.987060425939754e-26 + 0.0im


## 4. NormLayer Equivariance Test

In [5]:
norm_layer = NormLayer()

# NormLayer는 ||x||로 나누기 → g는 unitary이므로 ||gx|| = ||x||
# Path A: g * Norm(x)
out_norm = norm_layer([x])
A_norm = [g * out_norm[i] for i in eachindex(out_norm)]

# Path B: Norm(g*x)
B_norm = norm_layer([gx])

err_norm = sum(norm2(A_norm[i] - B_norm[i]) for i in eachindex(A_norm))
println("NormLayer equivariance error: ", err_norm)

NormLayer equivariance error: 9.558923916374134e-31 + 0.0im


## 5. BlockScalar_Layer Equivariance Test

In [6]:
r = 2
# nblocks = (NX÷r) * (NY÷r) * (NZ÷r) * (NT÷r)
n_vec = 2
bs = BlockScalar_Layer(x, n_vec, r)

# 입력: 2개 벡터
x2 = Initialize_pseudofermion_fields(U[1], "Wilson", nowing=true)
gauss_distribution_fermion!(x2)
gx2 = g * x2

# Path A: g * BlockScalar([x, x2])
out_bs = bs([x, x2])
A_bs = [g * out_bs[i] for i in eachindex(out_bs)]

# Path B: BlockScalar([g*x, g*x2])
B_bs = bs([gx, gx2])

err_bs = sum(norm2(A_bs[i] - B_bs[i]) for i in eachindex(A_bs))
println("BlockScalar_Layer equivariance error: ", err_bs)

BlockScalar_Layer equivariance error: 0.0 + 0.0im


## 6. DenseLinearLayer Equivariance Test

In [7]:
dl = DenseLinearLayer(1, 2, true)

# Path A: g * DenseLinear([x])
out_dl = dl([x])
A_dl = [g * out_dl[i] for i in eachindex(out_dl)]

# Path B: DenseLinear([g*x])
B_dl = dl([gx])

err_dl = sum(norm2(A_dl[i] - B_dl[i]) for i in eachindex(A_dl))
println("DenseLinearLayer equivariance error: ", err_dl)

DenseLinearLayer equivariance error: 1.0754705923537565e-27 + 0.0im


## 7. GramSchmidt_Layer Equivariance Test

In [8]:
gs = GramSchmidt_Layer()

# Path A: g * GS([x, x2])
out_gs = gs([x, x2])
A_gs = [g * out_gs[i] for i in eachindex(out_gs)]

# Path B: GS([g*x, g*x2])
# GramSchmidt는 inner_product 기반: <gx, gy> = <x,y> (unitary 불변)
gs2 = GramSchmidt_Layer()
B_gs = gs2([gx, gx2])

err_gs = sum(norm2(A_gs[i] - B_gs[i]) for i in eachindex(A_gs))
println("GramSchmidt_Layer equivariance error: ", err_gs)

GramSchmidt_Layer equivariance error: 7.182489313552526e-29 + 0.0im


## 8. Ritz_Layer Invariance Test

Ritz는 스칼라(고유값 합)를 출력하므로 invariant해야 함:  
$\text{Ritz}(x, D) \stackrel{?}{=} \text{Ritz}(gx, D')$

In [9]:
ritz = Ritz_Layer(D)
ritz_t = Ritz_Layer(D_transformed)

val_A = ritz([x, x2])
val_B = ritz_t([gx, gx2])

println("Ritz original:    ", val_A)
println("Ritz transformed: ", val_B)
println("Ritz invariance error: ", abs(val_A - val_B))

Ritz original:    2.8506264999624173
Ritz transformed: 2.8506264999624147
Ritz invariance error: 2.6645352591003757e-15


## 9. Full Model Equivariance Test

In [10]:
# 모델 구성
# fermionfield_Layer(x) — Int 인자 없이 호출해야 x를 그대로 저장
fe = fermionfield_Layer(x)
dl_m = DenseLinearLayer(1, 9, true)
pt1 = PTLayer(P, U, 9)
pt2 = PTLayer(P, U, 2)
norm_m = NormLayer()

model = [fe, dl_m, pt1, pt2, norm_m]

# Path A: g * Model(U)
out_model = model([U])
A_model = [g * out_model[i] for i in eachindex(out_model)]

# Path B: Model_transformed(U')
# fe_t.x = [g * fe.x[1]] で一貫性を保つ
fe_t = fermionfield_Layer(gx)
dl_m_t = DenseLinearLayer(1, 9, true)
dl_m_t.W = deepcopy(dl_m.W)
pt1_t = PTLayer(P, V, 9)
pt1_t.W = deepcopy(pt1.W)
pt2_t = PTLayer(P, V, 2)
pt2_t.W = deepcopy(pt2.W)
norm_m_t = NormLayer()

model_t = [fe_t, dl_m_t, pt1_t, pt2_t, norm_m_t]
B_model = model_t([V])

err_model = sum(norm2(A_model[i] - B_model[i]) for i in eachindex(A_model))
println("Full Model equivariance error: ", err_model)

Full Model equivariance error: 2.0488115052718272e-30 + 0.0im


## 결과 요약

In [11]:
println("========== Equivariance Test Results ==========")
println("PTLayer:            ", real(err) < 1e-20 ? "✓ PASS" : "✗ FAIL", "  (error = ", real(err), ")")
println("NormLayer:          ", real(err_norm) < 1e-20 ? "✓ PASS" : "✗ FAIL", "  (error = ", real(err_norm), ")")
println("BlockScalar_Layer:  ", real(err_bs) < 1e-20 ? "✓ PASS" : "✗ FAIL", "  (error = ", real(err_bs), ")")
println("DenseLinearLayer:   ", real(err_dl) < 1e-20 ? "✓ PASS" : "✗ FAIL", "  (error = ", real(err_dl), ")")
println("GramSchmidt_Layer:  ", real(err_gs) < 1e-20 ? "✓ PASS" : "✗ FAIL", "  (error = ", real(err_gs), ")")
println("Ritz_Layer:         ", abs(val_A - val_B) < 1e-10 ? "✓ PASS" : "✗ FAIL", "  (error = ", abs(val_A - val_B), ")")
println("Full Model:         ", real(err_model) < 1e-20 ? "✓ PASS" : "✗ FAIL", "  (error = ", real(err_model), ")")
println("================================================")

========== Equivariance Test Results ==========
PTLayer:            ✓ PASS  (error = 6.987060425939754e-26)
NormLayer:          ✓ PASS  (error = 9.558923916374134e-31)
BlockScalar_Layer:  ✓ PASS  (error = 0.0)
DenseLinearLayer:   ✓ PASS  (error = 1.0754705923537565e-27)
GramSchmidt_Layer:  ✓ PASS  (error = 7.182489313552526e-29)
Ritz_Layer:         ✓ PASS  (error = 2.6645352591003757e-15)
Full Model:         ✓ PASS  (error = 2.0488115052718272e-30)
